# Занятие 8. Вариационные автоэнкодеры (VAE)

## Цели занятия:
- Изучить вероятностную интерпретацию автоэнкодеров
- Реализовать VAE с репараметризацией (Reparameterization Trick)
- Научиться генерировать новые данные из латентного пространства
- Освоить интерполяцию между изображениями в latent space

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import random
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## Шаг 1. Загрузка данных (FashionMNIST)

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

---
## Шаг 2. Реализация VAE

In [ ]:
class VAE(nn.Module):
    """Variational Autoencoder.
    
    Key difference from AE:
    - Encoder outputs mean (mu) and log-variance (logvar)
    - Reparameterization trick: z = mu + std * epsilon
    """
    
    def __init__(self, latent_dim=16):
        super().__init__()
        
        # Encoder
        self.enc_fc1 = nn.Linear(28*28, 512)
        self.enc_fc2 = nn.Linear(512, 256)
        self.fc_mu = nn.Linear(256, latent_dim)      # Mean
        self.fc_logvar = nn.Linear(256, latent_dim)  # Log-variance
        
        # Decoder
        self.dec_fc1 = nn.Linear(latent_dim, 256)
        self.dec_fc2 = nn.Linear(256, 512)
        self.dec_out = nn.Linear(512, 28*28)
        
        self.latent_dim = latent_dim
    
    def encode(self, x):
        h = F.relu(self.enc_fc1(x))
        h = F.relu(self.enc_fc2(h))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """Reparameterization trick.
        
        z = mu + std * epsilon, where epsilon ~ N(0, 1)
        This allows backpropagation through sampling.
        """
        std = torch.exp(0.5 * logvar)  # std = sqrt(var)
        eps = torch.randn_like(std)    # epsilon ~ N(0, 1)
        return mu + eps * std
    
    def decode(self, z):
        h = F.relu(self.dec_fc1(z))
        h = F.relu(self.dec_fc2(h))
        return torch.sigmoid(self.dec_out(h))
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar

# Create model
model = VAE(latent_dim=16).to(device)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

---
## Шаг 3. Функция потерь VAE

In [ ]:
def vae_loss(x_hat, x, mu, logvar, beta=1.0):
    """VAE Loss = Reconstruction Loss + beta * KL Divergence.
    
    - Reconstruction: BCE between x_hat and x
    - KL: D_KL(N(mu, sigma) || N(0, 1))
    
    Args:
        beta: Weight for KL term (beta-VAE)
    """
    # Reconstruction loss (BCE)
    recon_loss = F.binary_cross_entropy(x_hat, x, reduction='sum')
    
    # KL divergence loss
    # D_KL = -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return recon_loss + beta * kl_loss, recon_loss, kl_loss

---
## Шаг 4. Обучение модели

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
history = {'recon_loss': [], 'kl_loss': [], 'total_loss': []}

print("="*50)
print("Training VAE")
print("="*50)

for epoch in range(15):
    model.train()
    total_recon, total_kl, total_loss = 0, 0, 0
    
    for images, _ in train_loader:
        images = images.view(-1, 28*28).to(device)
        
        optimizer.zero_grad()
        x_hat, mu, logvar = model(images)
        loss, recon, kl = vae_loss(x_hat, images, mu, logvar)
        loss.backward()
        optimizer.step()
        
        total_recon += recon.item()
        total_kl += kl.item()
        total_loss += loss.item()
    
    n_samples = len(train_dataset)
    history['recon_loss'].append(total_recon / n_samples)
    history['kl_loss'].append(total_kl / n_samples)
    history['total_loss'].append(total_loss / n_samples)
    
    print(f"Epoch {epoch+1}: Loss={total_loss/n_samples:.2f}, "
          f"Recon={total_recon/n_samples:.2f}, KL={total_kl/n_samples:.2f}")

---
## Шаг 5. Генерация новых изображений

In [ ]:
model.eval()

with torch.no_grad():
    # Sample from prior N(0, 1)
    z_sample = torch.randn(10, 16).to(device)
    generated = model.decode(z_sample)

# Visualize
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].cpu().view(28, 28), cmap='gray')
    ax.axis('off')

plt.suptitle('Generated Images from Random Noise', fontsize=14)
plt.tight_layout()
plt.savefig('vae_generated.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Шаг 6. Интерполяция в латентном пространстве

In [ ]:
# Get two latent vectors
model.eval()
with torch.no_grad():
    img1, _ = train_dataset[0]
    img2, _ = train_dataset[100]
    
    img1 = img1.view(-1, 28*28).to(device)
    img2 = img2.view(-1, 28*28).to(device)
    
    mu1, _ = model.encode(img1)
    mu2, _ = model.encode(img2)

# Interpolate
interpolations = []
with torch.no_grad():
    for alpha in torch.linspace(0, 1, 10):
        z_interp = alpha * mu1 + (1 - alpha) * mu2
        img_interp = model.decode(z_interp)
        interpolations.append(img_interp.cpu().view(28, 28))

# Visualize interpolation
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(interpolations[i], cmap='gray')
    ax.axis('off')

plt.suptitle('Latent Space Interpolation', fontsize=14)
plt.tight_layout()
plt.savefig('vae_interpolation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Шаг 7. Визуализация функции потерь

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['recon_loss'], label='Reconstruction', linewidth=2)
axes[0].plot(history['kl_loss'], label='KL Divergence', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('VAE Loss Components')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['total_loss'], label='Total Loss', color='red', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Total Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('vae_loss.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Домашнее задание

1. Поэкспериментировать с размером латентного пространства (8, 16, 32, 64)
2. Изменить коэффициент β в функции потерь (0.1, 1.0, 10.0)
3. Реализовать сверточный VAE (Conv2d + ConvTranspose2d)
4. Сравнить качество генерации VAE и обычного AE

---
## Контрольные вопросы

1. Зачем нужна репараметризация в VAE?
2. Что происходит при слишком высоком значении β?
3. Почему VAE генерирует более размытые изображения чем GAN?
4. Как интерпретировать KL-дивергенцию в контексте VAE?